# Puntos 7 y 8 - Validacion e informe de calidad

Este notebook ejecuta pruebas automaticas sobre el conjunto limpio y compara su estado con el archivo crudo. Las reglas se aplican sobre los campos destinados al analisis; los valores originales se conservan cuando son necesarios para la trazabilidad.

In [1]:
from __future__ import annotations

import re
import unicodedata
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent

RAW_PATH = ROOT / 'data' / 'raw' / 'establecimientos_diversificado_guatemala.csv'
CLEAN_PATH = ROOT / 'data' / 'processed' / 'establecimientos_diversificado_guatemala_limpio.csv'
DUPLICADOS_PATH = ROOT / 'data' / 'processed' / 'duplicados_candidatos.csv'
CAT_DIR = ROOT / 'data' / 'raw' / 'catalogos'
VALIDACION_PATH = ROOT / 'data' / 'processed' / 'resultados_validacion.csv'
INFORME_CSV_PATH = ROOT / 'data' / 'processed' / 'informe_calidad.csv'
CORRECCIONES_PATH = ROOT / 'data' / 'processed' / 'resumen_correcciones.csv'
INFORME_MD_PATH = ROOT / 'data' / 'processed' / 'informe_calidad.md'

BOOLEAN_COLUMNS = [
    'telefono_valido', 'distrito_formato_revisar',
    'departamental_difiere_departamento',
]
raw = pd.read_csv(RAW_PATH, dtype='string', keep_default_na=False)
clean_columns = pd.read_csv(CLEAN_PATH, nrows=0).columns.tolist()
clean_dtypes = {
    column: ('boolean' if column in BOOLEAN_COLUMNS else 'string')
    for column in clean_columns
    if column != 'fecha_extraccion'
}
clean = pd.read_csv(
    CLEAN_PATH,
    dtype=clean_dtypes,
    parse_dates=['fecha_extraccion'],
)
duplicados = pd.read_csv(DUPLICADOS_PATH, dtype='string')
cat_departamentos = pd.read_csv(CAT_DIR / 'departamentos.csv', dtype='string')
cat_municipios = pd.read_csv(CAT_DIR / 'municipios.csv', dtype='string')
cat_sectores = pd.read_csv(CAT_DIR / 'sectores.csv', dtype='string')
cat_modalidades = pd.read_csv(CAT_DIR / 'modalidades.csv', dtype='string')

print(f'Archivo crudo: {raw.shape[0]:,} registros y {raw.shape[1]} variables')
print(f'Archivo limpio: {clean.shape[0]:,} registros y {clean.shape[1]} variables')

Archivo crudo: 11,867 registros y 26 variables
Archivo limpio: 11,867 registros y 29 variables


## 7. Validacion automatica del conjunto limpio

Las pruebas comprueban duplicados exactos, espacios externos, formato de telefonos, catalogos territoriales, tipos de dato, categorias equivalentes e invalidos detectados durante el diagnostico. Una ejecucion se considera satisfactoria unicamente cuando todas las pruebas cumplen su regla.

In [2]:
PLACEHOLDERS = {'--', 'N/A', 'NULL', '-', '.', 'SIN DATO', 'NA', 'N.D.'}
CATEGORY_COLUMNS = [
    'departamento', 'municipio', 'nivel', 'sector', 'area', 'status',
    'modalidad', 'jornada', 'plan', 'departamental',
]
DISTRICT_PATTERN = r'(?:\d{2}-\d{3}|\d{2}-\d{2}-\d{3}|\d{2}-\d{2}-\d{4})'


def normalize_text_series(series: pd.Series) -> pd.Series:
    return (
        series.astype('string')
        .str.replace('\u00A0', ' ', regex=False)
        .str.replace(r'[\u200B-\u200D\uFEFF]', '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )


def fold_category(value: object) -> str:
    if pd.isna(value):
        return ''
    text = unicodedata.normalize('NFKD', str(value))
    text = ''.join(char for char in text if not unicodedata.combining(char))
    return re.sub(r'[^a-z0-9]+', '', text.lower())


def count_category_variants(frame: pd.DataFrame, columns: list[str]) -> int:
    variants = 0
    for column in columns:
        values = frame[column].dropna().astype('string')
        if values.empty:
            continue
        groups = values.groupby(values.map(fold_category)).nunique()
        variants += int((groups > 1).sum())
    return variants


def standardize_missing(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for column in result.columns:
        if column == 'fecha_extraccion':
            continue
        values = normalize_text_series(result[column])
        missing = values.eq('') | values.str.upper().isin(PLACEHOLDERS)
        if column == 'area':
            missing &= values.str.upper().ne('SIN ESPECIFICAR')
        result[column] = values.mask(missing, pd.NA)
    return result


def normalize_phone_series(series: pd.Series) -> pd.Series:
    values = series.astype('string').mask(series.astype('string').eq(''), pd.NA)
    values = values.str.replace(r'[\s\-\(\)]', '', regex=True)
    values = values.str.replace(r'^\+502', '', regex=True)
    digits = values.str.replace(r'\D', '', regex=True)
    return digits.mask(digits.eq(''), pd.NA)


def markdown_table(frame: pd.DataFrame) -> list[str]:
    columns = [str(column) for column in frame.columns]
    lines = [
        '| ' + ' | '.join(columns) + ' |',
        '| ' + ' | '.join(['---'] * len(columns)) + ' |',
    ]
    for row in frame.itertuples(index=False, name=None):
        values = [str(value).replace('|', '\\|').replace('\n', ' ') for value in row]
        lines.append('| ' + ' | '.join(values) + ' |')
    return lines

In [3]:
valid_departamentos = set(cat_departamentos['label'].str.upper())
valid_sectores = set(cat_sectores['label'].str.upper())
valid_modalidades = set(cat_modalidades['label'].str.upper())
valid_municipios = set(
    zip(
        cat_municipios['departamento'].str.upper(),
        cat_municipios['municipio'].str.upper(),
    )
)

text_columns = [
    column for column in clean.columns
    if column not in BOOLEAN_COLUMNS + ['fecha_extraccion']
]
space_issues = sum(
    int((clean[column].dropna() != normalize_text_series(clean[column].dropna())).sum())
    for column in text_columns
)
phone_format_issues = int((
    clean['telefono_normalizado'].notna()
    & ~clean['telefono_normalizado'].str.fullmatch(r'\d{8}', na=False)
).sum())
phone_flag_issues = int((
    clean['telefono_valido'].fillna(False)
    != clean['telefono_normalizado'].notna()
).sum())
invalid_departamentos = int((
    clean['departamento'].notna()
    & ~clean['departamento'].isin(valid_departamentos)
).sum())
invalid_municipios = sum(
    pair not in valid_municipios
    for pair in zip(clean['departamento'], clean['municipio'])
)
invalid_sectores = int((clean['sector'].notna() & ~clean['sector'].isin(valid_sectores)).sum())
invalid_modalidades = int((clean['modalidad'].notna() & ~clean['modalidad'].isin(valid_modalidades)).sum())
invalid_codigos = int((
    clean['codigo'].notna()
    & ~clean['codigo'].str.fullmatch(r'\d{2}-\d{2}-\d{4}-\d{2}', na=False)
).sum())
invalid_distritos = int((
    clean['distrito'].notna()
    & ~clean['distrito'].str.fullmatch(DISTRICT_PATTERN, na=False)
).sum())
invalid_nivel = int((clean['nivel'] != 'DIVERSIFICADO').fillna(True).sum())

type_errors = []
for column in text_columns:
    if not pd.api.types.is_string_dtype(clean[column].dtype):
        type_errors.append(column)
for column in BOOLEAN_COLUMNS:
    if not pd.api.types.is_bool_dtype(clean[column].dtype):
        type_errors.append(column)
if not pd.api.types.is_datetime64_any_dtype(clean['fecha_extraccion'].dtype):
    type_errors.append('fecha_extraccion')

known_invalid = (
    invalid_codigos + invalid_distritos + invalid_sectores
    + invalid_modalidades + invalid_nivel + phone_format_issues
    + int(clean['fecha_extraccion'].isna().sum())
)

resultados_validacion = pd.DataFrame([
    ('Duplicados exactos', 'No existen filas identicas', int(clean.duplicated().sum())),
    ('Espacios externos', 'Los textos no tienen espacios iniciales o finales', space_issues),
    ('Telefonos', 'El telefono normalizado tiene ocho digitos o es NA y su bandera coincide', phone_format_issues + phone_flag_issues),
    ('Departamentos', 'Todos los departamentos pertenecen al catalogo', invalid_departamentos),
    ('Municipios', 'Cada par departamento-municipio pertenece al catalogo', int(invalid_municipios)),
    ('Tipos de dato', 'Textos, fecha y banderas usan el tipo esperado', len(set(type_errors))),
    ('Categorias equivalentes', 'No hay categorias duplicadas por diferencias de escritura', count_category_variants(clean, CATEGORY_COLUMNS)),
    ('Invalidos del diagnostico', 'No quedan codigos, distritos, telefonos o categorias invalidas en los campos analiticos', known_invalid),
], columns=['prueba', 'regla', 'incumplimientos'])
resultados_validacion['resultado'] = resultados_validacion['incumplimientos'].eq(0).map({True: 'CUMPLE', False: 'NO CUMPLE'})
display(resultados_validacion)

fallos = resultados_validacion.loc[resultados_validacion['resultado'].eq('NO CUMPLE')]
assert fallos.empty, 'Fallaron las siguientes pruebas: ' + ', '.join(fallos['prueba'])
print(f'Resultado: {len(resultados_validacion)} de {len(resultados_validacion)} pruebas cumplen.')

,prueba,regla,incumplimientos,resultado
0,Duplicados exactos,No existen filas identicas,0,CUMPLE
1,Espacios externos,Los textos no tienen espacios iniciales o finales,0,CUMPLE
2,Telefonos,El telefono normalizado tiene ocho digitos o es NA y su bandera coincide,0,CUMPLE
3,Departamentos,Todos los departamentos pertenecen al catalogo,0,CUMPLE
4,Municipios,Cada par departamento-municipio pertenece al catalogo,0,CUMPLE
5,Tipos de dato,"Textos, fecha y banderas usan el tipo esperado",0,CUMPLE
6,Categorias equivalentes,No hay categorias duplicadas por diferencias de escritura,0,CUMPLE
7,Invalidos del diagnostico,"No quedan codigos, distritos, telefonos o categorias invalidas en los campos analiticos",0,CUMPLE


Resultado: 8 de 8 pruebas cumplen.


## 8. Informe de calidad antes y despues

La comparacion utiliza la definicion de cada metrica indicada en la guia. Los posibles duplicados se expresan como registros unicos involucrados en las parejas candidatas, no como numero de parejas.

In [4]:
raw_missing = standardize_missing(raw)
missing_before = int(raw_missing.isna().sum().sum())
missing_after = int(clean.isna().sum().sum())
missing_before_pct = 100 * missing_before / raw.size
missing_after_pct = 100 * missing_after / clean.size
variables_na_before = int(raw_missing.isna().any().sum())
variables_na_after = int(clean.isna().any().sum())

candidate_codes = set(duplicados['codigo_1'].dropna()) | set(duplicados['codigo_2'].dropna())
candidate_records = len(candidate_codes)
candidate_pairs = len(duplicados)

raw_phone_normalized = normalize_phone_series(raw['telefono'])
raw_invalid_phones = int((
    raw['telefono'].ne('')
    & ~raw_phone_normalized.str.fullmatch(r'\d{8}', na=False)
).sum())
raw_invalid_districts = int((
    raw['distrito'].ne('')
    & ~raw['distrito'].str.fullmatch(DISTRICT_PATTERN, na=False)
).sum())
format_before = int(raw_invalid_phones > 0) + int(raw_invalid_districts > 0)
format_after = int(phone_format_issues > 0) + int(invalid_distritos > 0)
type_before = int(not pd.api.types.is_datetime64_any_dtype(raw['fecha_extraccion'].dtype))
type_after = len(set(type_errors))
categories_before = count_category_variants(raw, CATEGORY_COLUMNS)
categories_after = count_category_variants(clean, CATEGORY_COLUMNS)

placeholder_corrections = 0
for column in raw.columns:
    if column == 'fecha_extraccion':
        continue
    values = normalize_text_series(raw[column])
    markers = values.ne('') & values.str.upper().isin(PLACEHOLDERS)
    if column == 'area':
        markers &= values.str.upper().ne('SIN ESPECIFICAR')
    placeholder_corrections += int(markers.sum())

total_corrected = placeholder_corrections + raw_invalid_phones + raw_invalid_districts
resumen_correcciones = pd.DataFrame([
    ('Marcadores de ausencia convertidos a NA', placeholder_corrections),
    ('Telefonos no normalizables retirados del campo analitico y marcados', raw_invalid_phones),
    ('Distritos incompletos convertidos a NA y marcados', raw_invalid_districts),
], columns=['tipo_correccion', 'registros_afectados'])

informe_calidad = pd.DataFrame([
    ('Registros', f'{len(raw):,}', f'{len(clean):,}', 'No se eliminaron filas.'),
    ('Variables', str(raw.shape[1]), str(clean.shape[1]), 'Se elimino municipio_consulta y se agregaron cuatro variables derivadas.'),
    ('Valores faltantes', f'{missing_before:,} ({missing_before_pct:.2f}%)', f'{missing_after:,} ({missing_after_pct:.2f}%)', 'Los marcadores se representan de forma uniforme como NA.'),
    ('Variables con NA', str(variables_na_before), str(variables_na_after), 'Se contabilizan variables con al menos un valor faltante.'),
    ('Duplicados exactos', str(int(raw.duplicated().sum())), str(int(clean.duplicated().sum())), 'No se encontraron filas identicas.'),
    ('Posibles duplicados', f'{candidate_records:,} registros en {candidate_pairs:,} parejas', f'{candidate_records:,} conservados; 0 corregidos; 0 fusionados', 'La decision permanece pendiente de revision manual; no se eliminaron registros automaticamente.'),
    ('Variables con formato inconsistente', str(format_before), str(format_after), f'Se trataron {raw_invalid_phones} telefonos y {raw_invalid_districts} distritos.'),
    ('Variables con tipo incorrecto', str(type_before), str(type_after), 'fecha_extraccion se convirtio de texto a fecha-hora UTC.'),
    ('Categorias inconsistentes', str(categories_before), str(categories_after), 'No se detectaron categorias equivalentes escritas de maneras distintas.'),
    ('Errores corregidos', f'{total_corrected:,} casos detectados', f'{total_corrected:,} casos tratados', 'El detalle se presenta por tipo de correccion.'),
], columns=['metrica', 'antes', 'despues', 'interpretacion'])

display(informe_calidad)
display(resumen_correcciones)

resultados_validacion.to_csv(VALIDACION_PATH, index=False, encoding='utf-8', lineterminator='\n')
informe_calidad.to_csv(INFORME_CSV_PATH, index=False, encoding='utf-8', lineterminator='\n')
resumen_correcciones.to_csv(CORRECCIONES_PATH, index=False, encoding='utf-8', lineterminator='\n')

report_lines = [
    '# Informe de calidad del conjunto MINEDUC',
    '',
    '## Resultados de validacion',
    '',
    *markdown_table(resultados_validacion),
    '',
    '## Comparacion antes y despues',
    '',
    *markdown_table(informe_calidad),
    '',
    '## Correcciones aplicadas',
    '',
    *markdown_table(resumen_correcciones),
    '',
    '## Observaciones',
    '',
    '- El numero de registros no cambio porque no habia duplicados exactos y los candidatos parciales requieren revision manual.',
    '- La columna municipio_consulta se elimino porque estaba completamente vacia.',
    '- Se agregaron telefono_normalizado, telefono_valido, distrito_formato_revisar y departamental_difiere_departamento.',
    '- El telefono original se conserva para auditoria; telefono_normalizado es el campo que debe utilizarse para analisis.',
    '- Los valores no verificables no se imputaron.',
]
INFORME_MD_PATH.write_text('\n'.join(report_lines), encoding='utf-8')

print(f'Validacion: {VALIDACION_PATH.relative_to(ROOT)}')
print(f'Informe CSV: {INFORME_CSV_PATH.relative_to(ROOT)}')
print(f'Informe Markdown: {INFORME_MD_PATH.relative_to(ROOT)}')

,metrica,antes,despues,interpretacion
0,Registros,"11,867","11,867",No se eliminaron filas.
1,Variables,26,29,Se elimino municipio_consulta y se agregaron cuatro variables derivadas.
2,Valores faltantes,"15,799 (5.12%)","5,196 (1.51%)",Los marcadores se representan de forma uniforme como NA.
3,Variables con NA,7,7,Se contabilizan variables con al menos un valor faltante.
4,Duplicados exactos,0,0,No se encontraron filas identicas.
5,Posibles duplicados,"8,414 registros en 13,221 parejas","8,414 conservados; 0 corregidos; 0 fusionados",La decision permanece pendiente de revision manual; no se eliminaron registros automaticamente.
6,Variables con formato inconsistente,2,0,Se trataron 248 telefonos y 70 distritos.
7,Variables con tipo incorrecto,1,0,fecha_extraccion se convirtio de texto a fecha-hora UTC.
8,Categorias inconsistentes,0,0,No se detectaron categorias equivalentes escritas de maneras distintas.
9,Errores corregidos,423 casos detectados,423 casos tratados,El detalle se presenta por tipo de correccion.


,tipo_correccion,registros_afectados
0,Marcadores de ausencia convertidos a NA,105
1,Telefonos no normalizables retirados del campo analitico y marcados,248
2,Distritos incompletos convertidos a NA y marcados,70


Validacion: data\processed\resultados_validacion.csv
Informe CSV: data\processed\informe_calidad.csv
Informe Markdown: data\processed\informe_calidad.md


## Resultado

El conjunto limpio cumple las reglas automaticas definidas para el punto 7. El informe del punto 8 deja documentados los cambios de estructura, los faltantes, los formatos corregidos y la decision de conservar los posibles duplicados hasta completar su revision manual.